# Ingestão Batch com CDF

In [ ]:
import uuid
from minio import Minio
from functools import partial
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *


MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppM7Class03") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.executor.cleanupOnShutdown", "true") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

## UTILS

### Declaring tables and locations

In [ ]:
# BRONZE
table_bronze = "clothes_batch_bronze"
location_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}"
location_checkpoint_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}/_checkpoint"
# SILVER
table_silver = "clothes_batch_silver"
location_silver = f"s3a://{BUCKET_SILVER}/delta/{table_silver}"
location_checkpoint_silver = f"s3a://{BUCKET_SILVER}/delta/{table_silver}/_checkpoint"
# GOLD
table_gold = "clothes_batch_gold"
location_gold = f"s3a://{BUCKET_GOLD}/delta/{table_gold}"
location_checkpoint_gold = f"s3a://{BUCKET_GOLD}/delta/{table_gold}/_checkpoint"

### Destroy environment

In [ ]:
# Configuração do cliente MinIO
minio_client = Minio(
    "minio:9000",  # endpoint do MinIO (ex: localhost:9000)
    access_key=MINIO_ACCESS_KEY,  # sua access key
    secret_key=MINIO_SECRET_KEY,  # sua secret key
    secure=False
)

spark.sql(f"DROP TABLE IF EXISTS {DATABASE}.{table_bronze}")
spark.sql(f"DROP TABLE IF EXISTS {DATABASE}.{table_silver}")
spark.sql(f"DROP TABLE IF EXISTS {DATABASE}.{table_gold}")

# Lista com os caminhos dos buckets
locations = [location_bronze, location_silver, location_gold]

# Loop para remover os objetos de cada local
for loc in locations:
    # Remove o esquema "s3a://" e extrai bucket e prefixo
    loc_no_scheme = loc.replace("s3a://", "")
    bucket, prefix = loc_no_scheme.split("/", 1)
    print(f"Removendo objetos do bucket '{bucket}' com prefixo '{prefix}'")
    
    try:
        # Lista os objetos sob o prefixo (remoção recursiva)
        objects = minio_client.list_objects(bucket, prefix=prefix, recursive=True)
        for obj in objects:
            minio_client.remove_object(bucket, obj.object_name)       
    except S3Error as err:
        print(f"Erro ao limpar o bucket '{bucket}' com prefixo '{prefix}': {err}")


In [ ]:
def merge_into_delta_table(cdf_df, batch_id, path):
    """
    Merges CDC data (cdf_df) into the table,
    using the 'id' and 'yearmonthday' fields as the key.
    If the table does not exist, it is created based on the cdf_df structure.
    
    To avoid the error of multiple rows corresponding to the same key, the source DataFrame
    is preprocessed to eliminate duplicates in the key.
    """
    # Tentar carregar a tabela; se não existir, cria uma tabela vazia
    try:
        layer_table = DeltaTable.forPath(spark, path)
    except Exception as e:
        # Tabela não existe. Criando uma tabela vazia
        empty_df = cdf_df.limit(0)
        empty_df.write.format("delta").mode("overwrite").save(path)
        layer_table = DeltaTable.forPath(spark, path)
    
    # Remove os update_preimage, mantendo insert, update_postimage e delete.
    filtered_df = cdf_df.filter(F.col("_change_type") != "update_preimage")
    
    # Elimina duplicatas na chave 'id' e 'yearmonthday' para evitar conflitos no merge.
    deduped_source = filtered_df.dropDuplicates(["id", "yearmonthday"])
    
    # Executa o MERGE:
    # - Se o registro existir e o _change_type for 'delete', ele é removido.
    # - Se o registro existir e o _change_type não for 'delete', ele é atualizado.
    # - Se não existir e o _change_type não for 'delete', ele é inserido.
    layer_table.alias("t").merge(
        source=deduped_source.alias("s"),
        condition="t.id = s.id AND t.yearmonthday = s.yearmonthday"
    ).whenMatchedDelete(condition="s._change_type = 'delete'") \
        .whenMatchedUpdateAll(condition="s._change_type <> 'delete'") \
        .whenNotMatchedInsertAll(condition="s._change_type <> 'delete'") \
        .execute()

def read_changes(table_origin, table_destination):

    tables_df = spark.sql(f"SHOW TABLES IN {DATABASE}")
    tables_list = [row.tableName for row in tables_df.collect()]
    
    if table_destination in tables_list:
        print(f"A tabela {table_destination} já existe no database {DATABASE}.")
        # Executa a query sem ORDER BY
        history_df = spark.sql(f"DESCRIBE HISTORY {DATABASE}.{table_origin}")
        
        # Ordena o DataFrame pela coluna "version" de forma decrescente
        latest_version = history_df.orderBy("version", ascending=False).first()["version"]
    
        # Define o startingVersion: pega a versão anterior, mas nunca menor que 0
        starting_version = max(int(latest_version) - 2, 0)
        print(f"Lendo mudanças de {table_origin} do CDC: startingVersion = {starting_version}, latest_version = {latest_version}")
        
        # Se a última versão for maior que o starting_version, usamos o endingVersion; 
        # caso contrário, omitimos essa opção.
        if int(latest_version) > starting_version:
             df_cdf = (
                spark.readStream.format("delta")
                     .option("readChangeFeed", "true")
                     .option("startingVersion", starting_version)
                     .option("endingVersion", latest_version)
                     .table(f"{DATABASE}.{table_origin}")
             )
        else:
             df_cdf = (
                spark.readStream.format("delta")
                     .option("readChangeFeed", "true")
                     .option("startingVersion", starting_version)
                     .table(f"{DATABASE}.{table_origin}")
             )
    else:
        # Se a tabela Silver não existir, lê tudo da Bronze (pode ser o primeiro carregamento completo)
        df_cdf = (
            spark.readStream.format("delta")
                 .option("readChangeFeed", "true")
                 .option("startingVersion", 0)
                 .table(f"{DATABASE}.{table_origin}")
        )

    return df_cdf

## Let's create a new table in the gold layer, from an existing one.

- Schema Enforcement/Evolution

In [ ]:
schema = "id int, data_venda timestamp, produto string, categoria string, quantidade long, preco_unitario double, preco_total double"

In [ ]:
location_raw = f"s3a://staging/batch/clothes"

In [ ]:
# Exemplo de leitura de dados em streaming (ajuste conforme sua fonte e esquema)
staging_df = (
    spark
    .readStream
    .schema(schema)
    .format("csv")
    .option("header", "true")
    .load(location_raw)  # Caminho dos arquivos CSV
)

In [ ]:
staging_df.printSchema()

<hr style="border: 2px solid black;">

<div style="background-color: #851d86; padding: 20px; border-radius: 8px; align: center">
  <h1 style="color: #fff; font-weight: bold;">JOB BRONZE</h1>
</div>

## Creating the BRONZE with CDC enabled (data change feed)

INGESTION BATCH DAILY - USING SPARK STREAMING WITH TRIGGER

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze} (
    id int,
    data_venda timestamp,
    produto string,
    categoria string,
    quantidade long,
    preco_unitario double,
    preco_total double,
    yearmonthday date
)
USING DELTA
LOCATION '{location_bronze}'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

In [ ]:
query = (
    staging_df
    .writeStream
    .outputMode("append")  # Adiciona novos registros sem sobrescrever os existentes
    .format("delta")
    .option("path", location_bronze)  # Diretório onde a tabela Delta foi criada
    .option("checkpointLocation", location_checkpoint_bronze)  # Local de checkpoint para garantir a consistência
    .trigger(availableNow=True)  # Processa todos os dados disponíveis e encerra o streaming
    .start()
)

print("Streaming of written starting...")

# Aguarda o término do processamento batch
query.awaitTermination()

print("Bronze (staging) table created and data ingested.")

## Exploring more Bronze

In [ ]:
spark.sql(f"SELECT * FROM {DATABASE}.{table_bronze} ORDER BY id").show(truncate=False)

In [ ]:
# Query to count total days of month there are on table
spark.sql(f"""
SELECT
    date_format(yearmonthday, 'yyyy-MM') AS month,
    COUNT(DISTINCT yearmonthday) AS total_days
FROM {DATABASE}.{table_bronze} 
GROUP BY date_format(yearmonthday, 'yyyy-MM')
""").show(truncate=False)

In [ ]:
spark.sql(f"SELECT COUNT(*) total_bronze FROM {DATABASE}.{table_bronze}").show(truncate=False)

<hr style="border: 2px solid black;">

<div style="background-color: #851d86; padding: 20px; border-radius: 8px; align: center">
  <h1 style="color: #fff; font-weight: bold;">JOB SILVER</h1>
</div>

## Creating the SILVER with CDC enabled (data change feed)

## Reading Changes from Bronze

In [ ]:
df_cdf = read_changes(table_origin=table_bronze, table_destination=table_silver)

In [ ]:
# df_cdf.show(truncate=False)

In [ ]:
# +----+-------------------+-----------+---------+----------+--------------+-----------+------------+----------------+---------------+-----------------------+
# |id  |data_venda         |produto    |categoria|quantidade|preco_unitario|preco_total|yearmonthday|_change_type    |_commit_version|_commit_timestamp      |
# +----+-------------------+-----------+---------+----------+--------------+-----------+------------+----------------+---------------+-----------------------+
# |1015|2025-01-01 00:00:00|Calça Jeans|Masculino|5         |354.81        |1774.05    |2025-01-01  |update_preimage |2              |2025-02-08 19:35:12.896|
# |1015|2025-01-01 00:00:00|Calça Jeans|Masculino|6         |354.81        |3000.0     |2025-01-01  |update_postimage|2              |2025-02-08 19:35:12.896|
# |1924|2025-01-01 00:00:00|Suéter     |Masculino|3         |439.68        |1319.04    |2025-01-01  |delete          |4              |2025-02-08 19:35:30.691|
# |10  |2025-01-01 00:00:00|Tenis      |Masculino|1         |600.0         |1555.0     |2025-01-01  |insert          |3              |2025-02-08 19:35:19.465|
# +----+-------------------+-----------+---------+----------+--------------+-----------+------------+----------------+---------------+-----------------------+

## Write Changes into Silver

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DATABASE}.{table_silver} (
    id int,
    data_venda date,
    produto string,
    categoria string,
    quantidade long,
    preco_unitario double,
    preco_total double,
    yearmonthday date
)
USING DELTA
LOCATION '{location_silver}'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

In [ ]:
%%time
merge_func = partial(merge_into_delta_table, path=location_silver)

query = (
    df_cdf
    .writeStream
    .format("delta")
    .foreachBatch(merge_func)
    .option("path", location_silver)
    .option("checkpointLocation", location_checkpoint_silver)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Silver (bronze) table created and data ingested.")

## Exploring more Silver

In [ ]:
spark.sql(f"SELECT * FROM {DATABASE}.{table_silver} ORDER BY id").show(truncate=False)

In [ ]:
# Query to count total days of month there are on Bronze
spark.sql(f"""
SELECT
    COUNT(*) as total_silver
FROM {DATABASE}.{table_bronze} 
""").show(truncate=False)

In [ ]:
# Query to count total days of month there are on Silver
spark.sql(f"""
SELECT
    COUNT(*) as total_silver
FROM {DATABASE}.{table_silver} 
""").show(truncate=False)

In [ ]:
# Query to count total days of month there are on table
spark.sql(f"""
SELECT
    date_format(yearmonthday, 'yyyy-MM') AS month,
    COUNT(DISTINCT yearmonthday) AS total_days
FROM {DATABASE}.{table_silver} 
GROUP BY date_format(yearmonthday, 'yyyy-MM')
""").show(truncate=False)

## Simulation of Changes in the Bronze Table

On 01/01/2025 there were changes in my transactional database (MySQL). I want to reflect these changes in my data lake

- Update
- Insert
- Delete

In [ ]:
# UPDATE
spark.sql(f"""
UPDATE {DATABASE}.{table_bronze}
SET quantidade = 6, preco_total = 3000.00
WHERE id = 1015
""")

In [ ]:
#INSERT
spark.sql(f"""
INSERT INTO {DATABASE}.{table_bronze} (id, data_venda, produto, categoria, quantidade, preco_unitario, preco_total, yearmonthday)
VALUES (
    10,
    DATE('2025-01-01'),
    'Tenis',
    'Masculino',
    1,
    600.0,
    1555.0,
    DATE('2025-01-01')
);
""")

In [ ]:
# DELETE
spark.sql(f"""
DELETE FROM {DATABASE}.{table_bronze}
WHERE id = 1924
""")

In [ ]:
spark.sql(f"""
SELECT *
FROM {DATABASE}.{table_bronze}
WHERE
    id IN(1015, 10, 1924)
ORDER BY
    id
""").show(truncate=False)

In [ ]:
spark.sql(f"""
SELECT *
FROM {DATABASE}.{table_silver}
WHERE
    id IN(1015, 10, 1924)
ORDER BY
    id
""").show(truncate=False)